In [0]:
%skip
# Databricks notebook source
# 04_train_register_SIMPLE.py
# SOLUCIÓN SIMPLE: Evita el error "For input string: None"
# Estrategia: Convertir a Pandas, limpiar, y volver a Spark

import mlflow
import mlflow.spark
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# CONFIGURACIÓN
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
GOLD_VOLUME_NAME = "gold_data"
gold_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{GOLD_VOLUME_NAME}/"

print("="*80)
print("🤖 ENTRENAMIENTO DE MODELO - VERSIÓN SIMPLE")
print("="*80)

# *******************************************************************
# PASO 1: CARGAR DATOS
# *******************************************************************
print("\n📂 Cargando features desde Gold...")
features_spark = spark.read.parquet(gold_path + "customer_features")

total_records = features_spark.count()
print(f"   Total registros: {total_records:,}")

# *******************************************************************
# PASO 2: CONVERTIR A PANDAS (Solución Simple)
# *******************************************************************
print("\n🔄 Convirtiendo a Pandas para limpieza...")

# Convertir a Pandas (funciona bien en datasets pequeños-medianos)
df_pandas = features_spark.toPandas()

print(f"   DataFrame Pandas creado: {df_pandas.shape}")

# *******************************************************************
# PASO 3: LIMPIEZA EN PANDAS (Más confiable)
# *******************************************************************
print("\n🧹 Limpiando datos...")

# Reemplazar strings problemáticos
for col in df_pandas.columns:
    if df_pandas[col].dtype == 'object':  # Columnas de texto
        # Reemplazar "None", "null", etc. con NaN
        df_pandas[col] = df_pandas[col].replace(['None', 'null', 'NULL', 'nan', 'NaN', ''], np.nan)
        
        # Intentar convertir a numérico
        df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce')

print("   ✅ Limpieza completada")

# *******************************************************************
# PASO 4: CREAR LABEL
# *******************************************************************
print("\n🎯 Creando variable objetivo 'is_premium'...")

# Llenar nulos en total_spent con 0
if 'total_spent' in df_pandas.columns:
    df_pandas['total_spent'] = df_pandas['total_spent'].fillna(0)
    
    # Calcular percentil 90
    threshold = df_pandas['total_spent'].quantile(0.90)
    
    # Crear label
    df_pandas['is_premium'] = (df_pandas['total_spent'] >= threshold).astype(int)
    
    print(f"   Umbral Premium (P90): {threshold:,.2f}")
    print(f"   Clientes Premium: {df_pandas['is_premium'].sum():,} ({df_pandas['is_premium'].mean()*100:.1f}%)")
else:
    print("   ⚠️  Columna 'total_spent' no encontrada, usando fallback")
    df_pandas['is_premium'] = 0

# *******************************************************************
# PASO 5: SELECCIONAR FEATURES
# *******************************************************************
print("\n📊 Seleccionando features...")

# Features candidatas
CANDIDATE_FEATURES = [
    'num_orders', 'num_orders_delivered', 'recency_days', 
    'avg_order_value', 'std_order_value', 'max_order_value',
    'avg_items_per_order', 'distinct_products_total',
    'avg_review_score', 'num_reviews', 'avg_delivery_days',
    'avg_installments_across_orders', 'max_payment_types_per_order',
    'spend_per_item', 'high_value_flag', 'loyal_customer_flag'
]

# Filtrar solo las que existen y son numéricas
FEATURES = []
for col in CANDIDATE_FEATURES:
    if col in df_pandas.columns:
        if pd.api.types.is_numeric_dtype(df_pandas[col]):
            FEATURES.append(col)
        else:
            print(f"   ⚠️  Ignorando '{col}' (no numérica)")
    else:
        print(f"   ⚠️  Columna '{col}' no existe")

print(f"\n   ✅ Features seleccionadas: {len(FEATURES)}")
print(f"   {FEATURES}")

# *******************************************************************
# PASO 6: IMPUTACIÓN DE NULOS
# *******************************************************************
print("\n🔧 Imputando valores faltantes...")

for col in FEATURES:
    null_count = df_pandas[col].isnull().sum()
    
    if null_count > 0:
        # Imputar con la mediana
        median_val = df_pandas[col].median()
        df_pandas[col] = df_pandas[col].fillna(median_val)
        print(f"   {col}: {null_count} nulos → mediana = {median_val:.2f}")

print("   ✅ Imputación completada")

# *******************************************************************
# PASO 7: VERIFICAR DATOS LIMPIOS
# *******************************************************************
print("\n✅ Verificación final...")

# Verificar que no hay NaN ni Inf
for col in FEATURES + ['is_premium']:
    nan_count = df_pandas[col].isnull().sum()
    inf_count = np.isinf(df_pandas[col]).sum() if pd.api.types.is_numeric_dtype(df_pandas[col]) else 0
    
    if nan_count > 0 or inf_count > 0:
        print(f"   ⚠️  {col}: {nan_count} NaN, {inf_count} Inf")
        # Llenar con 0 como último recurso
        df_pandas[col] = df_pandas[col].replace([np.inf, -np.inf], np.nan).fillna(0)

print("   ✅ Datos validados")

# *******************************************************************
# PASO 8: CONVERTIR DE VUELTA A SPARK
# *******************************************************************
print("\n🔄 Convirtiendo de vuelta a Spark...")

# Seleccionar solo columnas necesarias
df_model = df_pandas[FEATURES + ['is_premium']].copy()

# Convertir a Spark
features_clean = spark.createDataFrame(df_model)

print(f"   ✅ DataFrame Spark creado: {features_clean.count():,} registros")

# *******************************************************************
# PASO 9: ENSAMBLAR FEATURES
# *******************************************************************
print("\n🔧 Ensamblando features en vector...")

assembler = VectorAssembler(inputCols=FEATURES, outputCol="features")
final_df = assembler.transform(features_clean).select(
    "features",
    F.col("is_premium").alias("label")
)

print("   ✅ Ensamblaje exitoso")

# *******************************************************************
# PASO 10: SPLIT DE DATOS
# *******************************************************************
print("\n✂️  Dividiendo datos...")

train_df, val_df, test_df = final_df.randomSplit([0.70, 0.15, 0.15], seed=42)

print(f"   Train: {train_df.count():,} registros")
print(f"   Validation: {val_df.count():,} registros")
print(f"   Test: {test_df.count():,} registros")

# *******************************************************************
# PASO 11: ENTRENAR MODELO
# *******************************************************************
print("\n🤖 Entrenando modelo...")

mlflow.set_experiment("/Shared/experimentos/cliente_premium_simple")

with mlflow.start_run(run_name="logistic_regression_simple"):
    
    # Configurar modelo
    lr = LogisticRegression(
        featuresCol='features',
        labelCol='label',
        maxIter=10,
        regParam=0.01
    )
    
    print("   Entrenando Logistic Regression...")
    model = lr.fit(train_df)
    print("   ✅ Modelo entrenado")
    
    # *******************************************************************
    # PASO 12: EVALUAR
    # *******************************************************************
    print("\n📊 Evaluando modelo...")
    
    predictions_val = model.transform(val_df)
    
    evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    
    auc_val = evaluator.evaluate(predictions_val)
    
    print(f"   AUC Validation: {auc_val:.4f}")
    
    # *******************************************************************
    # PASO 13: LOG EN MLFLOW
    # *******************************************************************
    print("\n💾 Guardando en MLflow...")
    
    mlflow.log_param("num_features", len(FEATURES))
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_param("val_size", val_df.count())
    mlflow.log_param("features", str(FEATURES))
    mlflow.log_metric("auc_val", float(auc_val))
    
    # Guardar modelo
    mlflow.spark.log_model(
        model,
        "modelo_cliente_premium",
        registered_model_name="ClientePremiumSimpleModel"
    )
    
    run_id = mlflow.active_run().info.run_id
    print(f"   ✅ Run ID: {run_id}")

# *******************************************************************
# PASO 14: EVALUACIÓN FINAL EN TEST
# *******************************************************************
print("\n📊 Evaluación final en Test...")

predictions_test = model.transform(test_df)
auc_test = evaluator.evaluate(predictions_test)

print(f"   AUC Test: {auc_test:.4f}")

# Mostrar muestra de predicciones
print("\n   Muestra de predicciones:")
predictions_test.select("label", "prediction").groupBy("label", "prediction").count().show()

# *******************************************************************
# RESUMEN FINAL
# *******************************************************************
print("\n" + "="*80)
print("✅ ENTRENAMIENTO COMPLETADO")
print("="*80)
print(f"Modelo: ClientePremiumSimpleModel")
print(f"Run ID: {run_id}")
print(f"Features: {len(FEATURES)}")
print(f"AUC Validation: {auc_val:.4f}")
print(f"AUC Test: {auc_test:.4f}")
print("="*80)

# *******************************************************************
# PASO 15: GUARDAR LISTA DE FEATURES (Para inferencia)
# *******************************************************************
print("\n💾 Guardando lista de features...")

feature_list_df = spark.createDataFrame(
    [(i, feat) for i, feat in enumerate(FEATURES)],
    ["index", "feature_name"]
)

feature_list_df.write.mode("overwrite").parquet(gold_path + "model_features")

print(f"   ✅ Features guardadas en: {gold_path}model_features")
print("\n*** PROCESO COMPLETADO EXITOSAMENTE ***")

In [0]:
# Databricks notebook source
# 04_train_register_PANDAS.py
# SOLUCIÓN DEFINITIVA: Limpieza en Pandas (100% confiable)

import mlflow
import mlflow.spark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

# *******************************************************************
# CONFIGURACIÓN
# *******************************************************************
CATALOG_NAME = "olist"
SCHEMA_NAME = "olist_csv"
GOLD_VOLUME_NAME = "gold_data"
gold_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{GOLD_VOLUME_NAME}/"

print("="*80)
print("🤖 ENTRENAMIENTO - VERSIÓN PANDAS (Solución Definitiva)")
print("="*80)

# *******************************************************************
# PASO 1: CARGAR DATOS EN SPARK
# *******************************************************************
print("\n📂 Cargando features desde Gold...")
features_spark = spark.read.parquet(gold_path + "customer_features")

initial_count = features_spark.count()
print(f"   Registros: {initial_count:,}")
print(f"   Columnas: {len(features_spark.columns)}")

# *******************************************************************
# PASO 2: CONVERTIR A PANDAS (LIMPIEZA CONFIABLE)
# *******************************************************************
print("\n🔄 Convirtiendo a Pandas para limpieza...")
df = features_spark.toPandas()
print(f"   DataFrame Pandas: {df.shape}")

# *******************************************************************
# PASO 3: LIMPIEZA EXHAUSTIVA EN PANDAS
# *******************************************************************
print("\n🧹 Limpiando datos en Pandas...")

# Lista de valores que representan NULL
NULL_VALUES = ['None', 'null', 'NULL', 'nan', 'NaN', 'none', 'NONE', '', 'NA', 'N/A', 'na', '<NA>']

# Reemplazar en TODAS las columnas
print("   🔄 Reemplazando valores nulos...")
for col in df.columns:
    # Si es object/string, reemplazar strings problemáticos
    if df[col].dtype == 'object':
        df[col] = df[col].replace(NULL_VALUES, np.nan)
        # Intentar convertir a numérico
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Si es numérico, reemplazar inf
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)

print("   ✅ Valores nulos reemplazados")

# *******************************************************************
# PASO 4: CREAR LABEL
# *******************************************************************
print("\n🎯 Creando label 'is_premium'...")

if 'total_spent' in df.columns:
    # Asegurar que total_spent es numérico
    df['total_spent'] = pd.to_numeric(df['total_spent'], errors='coerce').fillna(0)
    
    # Calcular threshold
    threshold = df['total_spent'].quantile(0.90)
    
    # Crear label
    df['is_premium'] = (df['total_spent'] >= threshold).astype(int)
    
    premium_count = df['is_premium'].sum()
    print(f"   Umbral P90: ${threshold:,.2f}")
    print(f"   Clientes premium: {premium_count:,} ({premium_count/len(df)*100:.1f}%)")
else:
    print("   ⚠️  'total_spent' no encontrada, usando fallback")
    df['is_premium'] = 0

# *******************************************************************
# PASO 5: SELECCIONAR FEATURES CLAVE
# *******************************************************************
print("\n📊 Seleccionando features...")

CANDIDATE_FEATURES = [
    'num_orders',
    'recency_days',
    'avg_order_value',
    'total_spent',
    'distinct_products_total',
    'avg_review_score',
    'avg_delivery_days',
    'high_value_flag',
    'loyal_customer_flag',
    'num_orders_delivered'
]

# Filtrar solo las que existen y son numéricas
FEATURES = []
for col in CANDIDATE_FEATURES:
    if col in df.columns:
        # Asegurar que es numérica
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
        if pd.api.types.is_numeric_dtype(df[col]):
            FEATURES.append(col)
        else:
            print(f"   ⚠️  Ignorando '{col}' (no numérica)")
    else:
        print(f"   ⚠️  '{col}' no existe")

print(f"\n   ✅ Features seleccionadas: {len(FEATURES)}")
for feat in FEATURES:
    print(f"      - {feat}")

if len(FEATURES) == 0:
    raise ValueError("❌ No hay features válidas!")

# *******************************************************************
# PASO 6: IMPUTACIÓN EN PANDAS
# *******************************************************************
print("\n🔧 Imputando valores nulos...")

for col in FEATURES:
    null_count = df[col].isnull().sum()
    
    if null_count > 0:
        # Imputar con mediana
        median_val = df[col].median()
        
        if pd.isna(median_val):
            median_val = 0.0
            print(f"   ⚠️  {col}: toda la columna nula → 0.0")
        else:
            print(f"   {col}: {null_count} nulos → mediana = {median_val:.2f}")
        
        df[col] = df[col].fillna(median_val)

print("   ✅ Imputación completada")

# *******************************************************************
# PASO 7: VALIDACIÓN EXHAUSTIVA
# *******************************************************************
print("\n✅ Validación final...")

for col in FEATURES + ['is_premium']:
    # Verificar NaN
    nan_count = df[col].isnull().sum()
    if nan_count > 0:
        print(f"   ⚠️  {col}: {nan_count} NaN → rellenando con 0")
        df[col] = df[col].fillna(0.0)
    
    # Verificar Inf
    if pd.api.types.is_numeric_dtype(df[col]):
        inf_count = np.isinf(df[col]).sum()
        if inf_count > 0:
            print(f"   ⚠️  {col}: {inf_count} Inf → rellenando con 0")
            df[col] = df[col].replace([np.inf, -np.inf], 0.0)

print("   ✅ Datos validados")

# *******************************************************************
# PASO 8: FORZAR TIPOS EXPLÍCITOS
# *******************************************************************
print("\n🔧 Forzando tipos de datos...")

for col in FEATURES:
    df[col] = df[col].astype('float64')

df['is_premium'] = df['is_premium'].astype('int32')

print("   ✅ Tipos forzados correctamente")

# Verificar tipos finales
print("\n   📊 Tipos finales:")
for col in FEATURES + ['is_premium']:
    print(f"      {col:30s}: {df[col].dtype}")

# *******************************************************************
# PASO 9: CONVERTIR A SPARK CON ESQUEMA EXPLÍCITO
# *******************************************************************
print("\n🔄 Convirtiendo a Spark con esquema explícito...")

# Crear esquema explícito
schema_fields = [StructField(col, DoubleType(), False) for col in FEATURES]
schema_fields.append(StructField('is_premium', IntegerType(), False))
schema = StructType(schema_fields)

# Seleccionar solo columnas necesarias
df_model = df[FEATURES + ['is_premium']].copy()

# Convertir a Spark
ml_data = spark.createDataFrame(df_model, schema=schema)

spark_count = ml_data.count()
print(f"   ✅ DataFrame Spark: {spark_count:,} registros")

# Mostrar esquema
print("\n   📊 Esquema Spark:")
ml_data.printSchema()

# *******************************************************************
# PASO 10: ENSAMBLAR FEATURES
# *******************************************************************
print("\n🔧 Ensamblando features...")

assembler = VectorAssembler(
    inputCols=FEATURES,
    outputCol="features",
    handleInvalid="skip"
)

final_df = assembler.transform(ml_data).select(
    "features",
    F.col("is_premium").alias("label")
)

final_count = final_df.count()
print(f"   ✅ Dataset final: {final_count:,} registros")

# *******************************************************************
# PASO 11: SPLIT DE DATOS
# *******************************************************************
print("\n✂️  Dividiendo datos...")

train_df, test_df = final_df.randomSplit([0.80, 0.20], seed=42)

train_count = train_df.count()
test_count = test_df.count()

print(f"   Train: {train_count:,} ({train_count/final_count*100:.1f}%)")
print(f"   Test: {test_count:,} ({test_count/final_count*100:.1f}%)")

print("\n   📊 Distribución Train:")
train_df.groupBy("label").count().show()

# *******************************************************************
# PASO 12: ENTRENAR MODELO
# *******************************************************************
print("\n🤖 Entrenando modelo...")

mlflow.set_experiment("/Shared/experimentos/cliente_premium_pandas")

with mlflow.start_run(run_name="logistic_regression_pandas"):
    
    lr = LogisticRegression(
        featuresCol='features',
        labelCol='label',
        maxIter=10,
        regParam=0.1,
        elasticNetParam=0.5,
        standardization=False
    )
    
    print("   🔄 Entrenando Logistic Regression...")
    model = lr.fit(train_df)
    print("   ✅ Modelo entrenado exitosamente!")
    
    # *******************************************************************
    # PASO 13: EVALUAR
    # *******************************************************************
    print("\n📊 Evaluando modelo...")
    
    predictions = model.transform(test_df)
    
    # AUC
    evaluator_auc = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )
    auc_score = evaluator_auc.evaluate(predictions)
    
    # Accuracy
    evaluator_acc = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )
    accuracy = evaluator_acc.evaluate(predictions)
    
    # Precision y Recall
    precision = evaluator_acc.evaluate(predictions, {evaluator_acc.metricName: "weightedPrecision"})
    recall = evaluator_acc.evaluate(predictions, {evaluator_acc.metricName: "weightedRecall"})
    f1 = evaluator_acc.evaluate(predictions, {evaluator_acc.metricName: "f1"})
    
    print(f"   AUC-ROC: {auc_score:.4f}")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   F1-Score: {f1:.4f}")
    
    print("\n   📊 Matriz de confusión:")
    predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()
    
    # *******************************************************************
    # PASO 14: GUARDAR METADATA EN MLFLOW
    # *******************************************************************
    print("\n💾 Guardando en MLflow...")
    
    mlflow.log_param("num_features", len(FEATURES))
    mlflow.log_param("features", str(FEATURES))
    mlflow.log_param("train_size", train_count)
    mlflow.log_param("test_size", test_count)
    mlflow.log_param("threshold_p90", float(threshold))
    
    mlflow.log_metric("auc_roc", float(auc_score))
    mlflow.log_metric("accuracy", float(accuracy))
    mlflow.log_metric("precision", float(precision))
    mlflow.log_metric("recall", float(recall))
    mlflow.log_metric("f1_score", float(f1))
    
    # Guardar lista de features
    features_text = "\n".join([f"{i+1}. {feat}" for i, feat in enumerate(FEATURES)])
    mlflow.log_text(features_text, "features.txt")
    
    # Guardar coeficientes
    coef_text = "COEFICIENTES DEL MODELO\n" + "="*50 + "\n\n"
    for i in range(len(FEATURES)):
        coef_text += f"{FEATURES[i]:30s}: {model.coefficients[i]:>10.6f}\n"
    coef_text += f"\n{'Intercept':30s}: {model.intercept:>10.6f}\n"
    mlflow.log_text(coef_text, "coefficients.txt")
    
    run_id = mlflow.active_run().info.run_id
    print(f"   ✅ Run ID: {run_id}")

# *******************************************************************
# PASO 15: GUARDAR EN GOLD LAYER
# *******************************************************************
print("\n💾 Guardando artefactos en Gold Layer...")

# Features
feature_df = spark.createDataFrame(
    [(i, feat) for i, feat in enumerate(FEATURES)],
    ["index", "feature_name"]
)
feature_df.write.mode("overwrite").parquet(gold_path + "model_features_final")

# Coeficientes
coef_data = [(FEATURES[i], float(model.coefficients[i])) for i in range(len(FEATURES))]
coef_data.append(("_intercept_", float(model.intercept)))
coef_df = spark.createDataFrame(coef_data, ["feature_name", "coefficient"])
coef_df.write.mode("overwrite").parquet(gold_path + "model_coefficients_final")

# Métricas
metrics_df = spark.createDataFrame([
    ("auc_roc", float(auc_score)),
    ("accuracy", float(accuracy)),
    ("precision", float(precision)),
    ("recall", float(recall)),
    ("f1_score", float(f1)),
    ("threshold_p90", float(threshold)),
    ("num_features", float(len(FEATURES)))
], ["metric_name", "metric_value"])
metrics_df.write.mode("overwrite").parquet(gold_path + "model_metrics_final")

print(f"   ✅ Guardado en: {gold_path}")

# *******************************************************************
# RESUMEN FINAL
# *******************************************************************
print("\n" + "="*80)
print("✅ ENTRENAMIENTO COMPLETADO EXITOSAMENTE")
print("="*80)
print(f"\n🆔 Run ID: {run_id}")
print(f"🎯 Features utilizadas: {len(FEATURES)}")
print(f"\n📊 Datos:")
print(f"   Training:   {train_count:>10,} registros ({train_count/final_count*100:.1f}%)")
print(f"   Test:       {test_count:>10,} registros ({test_count/final_count*100:.1f}%)")
print(f"\n📈 Métricas de Test:")
print(f"   AUC-ROC:    {auc_score:>10.4f}")
print(f"   Accuracy:   {accuracy:>10.4f}")
print(f"   Precision:  {precision:>10.4f}")
print(f"   Recall:     {recall:>10.4f}")
print(f"   F1-Score:   {f1:>10.4f}")
print(f"\n📁 Artefactos guardados en:")
print(f"   {gold_path}")
print(f"\n💡 Features: {', '.join(FEATURES[:5])}...")
print("="*80)
print("\n🎉 ¡Modelo listo para producción!")